        # 🧭 L09　沒有標準答案的學習：PCA 與 K-means
        **統計冒險之旅 2026**　｜　Day 4（10/05 一）🚩 登頂日　｜　關卡　｜　🏅 100 XP

        📖 ISLP Ch12；資料：勇者咖啡會員


        ### 🎯 這一關你會學到
        - 標準化後做 PCA、解釋變異比例與雙標圖
- K-means 分群、肘部法選 K
- 幫每一群取名字

        ### 🧭 闖關方式
        1. 先按下方「🧰 魔法工具箱」那一格左邊的 ▶（第一次執行 Colab 會花幾秒鐘連線）。
        2. 依序閱讀說明、執行範例、完成每個「🎯 任務」，再執行它下面的「檢查」格。
        3. 看到 ✅ 就往下一個任務；看到 ❌ 就依提示修改，再重新執行任務格與檢查格。
        4. 全部通過後，執行最下面的「🔑 通關密語」格，把密語貼回 [入口網頁](https://johnnychao.github.io/stats-quest-2026/)。

        > 💾 建議先點選「檔案 → 在雲端硬碟中儲存副本」，你的進度才會留在自己的 Google 雲端硬碟。
        > 🎲 這門課的答案常常是小數：任務會告訴你要把答案存進哪個變數，檢查時允許小小的誤差；切分、抽樣、模型請照題目用 `random_state=42`。

In [ ]:
#@title 🧰 魔法工具箱：先在右邊填「暱稱」，再按左邊的 ▶ 執行這一格 { display-mode: "form" }
暱稱 = "" #@param {type:"string"}
# ======================================================================
#  統計冒險之旅 2026 · 關卡檢查工具（看不懂沒關係，這一格不是今天的功課 😉）
# ======================================================================
import hashlib, unicodedata, io, sys, re, contextlib, traceback, builtins, math, warnings
warnings.filterwarnings("ignore")

_LEVEL = "L09"
_COURSE_NAMESPACE = "stats-quest-2026-datama"
_PREFIX = "SQ"
_TASKS = ["9-1", "9-2", "9-3", "9-4", "9-5"]
_XP_EACH = 20
_CHECKS = {}
_PASSED = builtins.__dict__.setdefault("_sq_" + _LEVEL, {})
_HINTS = {}

def _norm_name(s):
    return re.sub(r"\s+", "", unicodedata.normalize("NFKC", str(s))).lower()

def _squash(s):
    return re.sub(r"\s+", "", str(s))

def 出現(out, *subs):
    """輸出中是否（忽略空白）包含所有片段"""
    o = _squash(out)
    return all(_squash(x) in o for x in subs)

def 數字們(out):
    """抓出輸出裡所有的數字（float）"""
    return [float(x) for x in re.findall(r"-?\d+(?:\.\d+)?", str(out))]

# ---------------- 判分器 2.0 ----------------
class _Miss(Exception):
    pass

def 抓變數(ns, name, 型別=None):
    """從任務格執行後的變數取值；沒有就給友善訊息。"""
    if name not in ns:
        raise _Miss(f"我找不到變數 {name}，請確認你有把答案存進名字叫 {name} 的變數（大小寫要一樣）。")
    v = ns[name]
    if 型別 is not None and not isinstance(v, 型別):
        raise _Miss(f"{name} 的型別看起來不對（目前是 {type(v).__name__}）。")
    return v

def _num(v):
    try:
        import numpy as _np
        if hasattr(v, "item"): v = v.item()
    except Exception:
        pass
    return float(v)

def 約等於(v, 目標, 容差=None, 相對=0.01):
    """數值容差：|v-目標| <= 容差（預設為 目標 的 1%，且至少 1e-9）"""
    try:
        x = _num(v)
    except Exception:
        return False
    if x != x:   # NaN
        return False
    tol = 容差 if 容差 is not None else max(abs(目標) * 相對, 1e-9)
    return abs(x - 目標) <= tol

def 資料框像(obj, 列=None, 欄=None, 含欄位=None, 種類="DataFrame"):
    """檢查 DataFrame / Series：列數、欄數、必須包含的欄位；回傳 (ok, 訊息)"""
    import pandas as _pd
    if 種類 == "DataFrame" and not isinstance(obj, _pd.DataFrame):
        return False, f"這應該是一個 DataFrame（目前是 {type(obj).__name__}）。"
    if 種類 == "Series" and not isinstance(obj, _pd.Series):
        return False, f"這應該是一個 Series（目前是 {type(obj).__name__}）。"
    if 列 is not None and len(obj) != 列:
        return False, f"列數應該是 {列}，目前是 {len(obj)}。"
    if 欄 is not None and getattr(obj, "shape", (0, 0))[1] != 欄:
        return False, f"欄數應該是 {欄}，目前是 {obj.shape[1]}。"
    if 含欄位:
        cols = list(obj.columns) if hasattr(obj, "columns") else list(obj.index)
        missing = [c for c in 含欄位 if c not in cols]
        if missing:
            return False, "缺少欄位：" + "、".join(map(str, missing))
    return True, ""


class _NeedMoreInput(Exception):
    pass

_BUILTIN_NAMES = ("sum", "list", "dict", "set", "str", "int", "float", "max", "min", "len",
                  "print", "type", "range", "sorted", "abs", "round", "tuple", "map", "filter",
                  "open", "format", "all", "any", "zip", "bool", "next", "chr", "ord", "id")

_HIST = builtins.__dict__.setdefault("_sq_hist", [])
def _on_pre_run(*args):
    try:
        info = args[0]
        src = getattr(info, "raw_cell", None)
        if isinstance(src, str):
            _HIST.append(src)
    except Exception:
        pass
try:
    _ip = get_ipython()
    if not builtins.__dict__.get("_sq_hooked"):
        _ip.events.register("pre_run_cell", _on_pre_run)
        builtins.__dict__["_sq_hooked"] = True
except Exception:
    pass

def _history():
    try:
        ip = get_ipython()
        h = list(ip.user_ns.get("In") or ip.user_ns.get("_ih") or [])
    except Exception:
        h = list(globals().get("In") or [])
    return [c for c in (h + list(_HIST)) if isinstance(c, str)]

_CALL = re.compile(r"\s*(檢查|通關密語|全部檢查)\s*\(")

def _clean_cell(cell):
    return "\n".join(ln for ln in cell.splitlines() if not _CALL.match(ln))

def _is_mine(cell):
    s = cell.strip()
    if not s:
        return False
    if "#@title" in s or "任務定義(" in s or "_sq_" in s:
        return False
    if _CALL.match(s):
        return False
    return True

def _find_cells(tid):
    marker = "# 🎯 任務 " + tid
    marked = free = None
    im = ifree = -1
    for i, cell in enumerate(_history()):
        if not _is_mine(cell):
            continue
        if marker in cell:
            marked, im = cell, i
        elif "🎯 任務" not in cell:
            free, ifree = cell, i
    return marked, im, free, ifree

def _describe(src):
    body = [ln for ln in src.splitlines() if ln.strip() and not ln.strip().startswith("#")]
    if not body:
        return "（空白）"
    first = body[0].strip()
    return ("%s%s（共 %d 行）" % (first[:52], "…" if len(first) > 52 else "", len(body)))

def _fig_info(_plt):
    out = []
    try:
        for n in _plt.get_fignums():
            f = _plt.figure(n)
            for ax in f.get_axes():
                out.append(dict(title=ax.get_title() or "", xlabel=ax.get_xlabel() or "", ylabel=ax.get_ylabel() or "",
                                n_lines=len(ax.lines), n_patches=len(ax.patches), n_collections=len(ax.collections),
                                legend=bool(ax.get_legend())))
    except Exception:
        pass
    return out

def _make_runner(src):
    def run(*inputs):
        feed = iter([str(x) for x in inputs])
        buf = io.StringIO()
        try:
            ns = dict(get_ipython().user_ns)
        except Exception:
            ns = dict(globals())
        run.shadowed = []
        for _n in _BUILTIN_NAMES:
            _b = getattr(builtins, _n, None)
            if _n in ns and _b is not None and ns[_n] is not _b:
                ns.pop(_n, None)
                run.shadowed.append(_n)
        def _fake_input(prompt=""):
            try:
                return next(feed)
            except StopIteration:
                raise _NeedMoreInput()
        ns["input"] = _fake_input
        ns["__name__"] = "__main__"
        try:
            import matplotlib
            import matplotlib.pyplot as _plt
            _plt.close("all"); _orig_show = _plt.show; _plt.show = lambda *a, **k: None
        except Exception:
            _plt = None
        run.figs = []
        try:
            with contextlib.redirect_stdout(buf):
                exec(compile(src, "<任務 " + _LEVEL + ">", "exec"), ns)
        finally:
            if _plt is not None:
                run.figs = _fig_info(_plt)
                _plt.show = _orig_show
                _plt.close("all")
        return buf.getvalue(), ns
    run.src = src
    run.figs = []
    return run

def 任務定義(tid, fn, 提示=""):
    _CHECKS[tid] = fn
    _HINTS[tid] = 提示

def _fix_shadowed():
    try:
        ns = get_ipython().user_ns
    except Exception:
        ns = globals()
    bad = []
    for n in _BUILTIN_NAMES:
        b = builtins.__dict__.get(n)
        if b is not None and n in ns and ns[n] is not b:
            del ns[n]
            bad.append(n)
    return bad

def _progress():
    done = 0
    total = 0
    for t in _TASKS:
        total += 1
        if _PASSED.get(t):
            done += 1
    bar = "■" * done + "□" * (total - done)
    return f"[{bar}] {done}/{total}"

def _run_check(tid, src):
    run = _make_runner(src)
    try:
        result = _CHECKS[tid](run)
    except _NeedMoreInput:
        return False, "你的程式呼叫 input() 的次數比題目預期的多，請檢查輸入的次數。", []
    except _Miss as e:
        return False, str(e), getattr(run, "shadowed", [])
    except Exception:
        tb = traceback.format_exc().strip().splitlines()[-1]
        return False, "程式執行時發生錯誤 → " + tb, getattr(run, "shadowed", [])
    ok, extra = (result, "") if isinstance(result, bool) else result
    return ok, extra, getattr(run, "shadowed", [])

def _pass(tid):
    first = not _PASSED.get(tid)
    _PASSED[tid] = True
    print(f"✅ 任務 {tid} 通過！{'+' + str(_XP_EACH) + ' XP ' if first else ''}{_progress()}")

def 檢查(tid):
    _shadow = _fix_shadowed()
    tid = builtins.str(tid)
    if tid not in _CHECKS:
        print(f"⚠️ 找不到任務 {tid} 的檢查設定。"); return
    marked, im, free, ifree = _find_cells(tid)
    if marked is None and free is None:
        print(f"❌ 這次執行階段裡，我找不到你寫的程式。")
        print(f"   👉 請先按「# 🎯 任務 {tid}」那一格左邊的 ▶ 執行它，再執行這一格。")
        print("   （如果剛剛重新啟動過執行階段，上面每一格都要重跑一次，包含最上面的魔法工具箱）")
        return
    order = []
    if marked is not None:
        order.append(("標記", marked))
    if free is not None and ifree > im:
        order.append(("最後執行", free))
    if not order:
        order = [("最後執行", free)]
    tried = []
    for kind, src in order:
        ok, extra, shadowed = _run_check(tid, _clean_cell(src))
        tried.append((kind, src, extra, shadowed))
        if ok:
            _pass(tid)
            if extra:
                print("   💬 " + str(extra))
            if _shadow:
                print(f"   ℹ️ 你之前把內建名稱 {'、'.join(_shadow)} 拿來當變數名了，我已經幫你還原。")
                print("      建議換個名字（例如 total、items），不然後面的程式會出現很難懂的錯誤。")
            if kind == "最後執行":
                print(f"   ℹ️ 你的程式最上面少了「# 🎯 任務 {tid}」那一行，我是用你最後執行的那一格判分的。")
                print("      把那一行加回去，之後的檢查會更準確。")
            if shadowed:
                print(f"   ℹ️ 你之前把內建名稱 {'、'.join(shadowed)} 拿來當變數名了，判分時我先幫你還原。")
            if all(_PASSED.get(t) for t in _TASKS):
                print("🏆 本關所有任務都完成了！請執行最下面的「通關密語」那一格。")
            return
    kind, src, extra, shadowed = tried[0]
    print(f"❌ 任務 {tid} 還沒通過。{_progress()}")
    if extra:
        print("   💬 " + str(extra))
    if _HINTS.get(tid):
        print("   💡 提示：" + _HINTS[tid])
    _sh = _shadow + [n for n in shadowed if n not in _shadow]
    if _sh:
        print(f"   ⚠️ 你把內建名稱 {'、'.join(_sh)} 拿來當變數名了（我已還原），這會造成很難懂的錯誤，請改名後重跑那一格。")
    print("   🔎 我判分的是這一段程式：" + _describe(src))
    print(f"      如果這不是你剛剛寫的版本 → 確認第一行的「# 🎯 任務 {tid}」有保留，並重新執行那一格，再按檢查。")

def 全部檢查():
    """出錯或重新啟動執行階段後，重跑完所有任務格，再用這個一次驗收整關。"""
    _fix_shadowed()
    print(f"🔁 重新檢查 {_LEVEL} 的 {len(_TASKS)} 個任務…")
    todo = []
    for t in _TASKS:
        marked, im, free, ifree = _find_cells(t)
        if marked is None and free is None:
            todo.append(t)
            continue
        檢查(t)
    if todo:
        print("⏭️ 這次還沒執行過的任務：" + "、".join(todo))
        print("   先按那幾格左邊的 ▶ 執行，再回來執行 全部檢查()。")

def 通關密語():
    _fix_shadowed()
    missing = [t for t in _TASKS if not _PASSED.get(t)]
    if missing:
        print("🔒 還有任務未通過：" + "、".join(missing) + "　完成後再來拿密語吧！")
        return
    name = 暱稱.strip() if isinstance(暱稱, str) else ""
    if not name:
        name = input("請輸入你在入口網頁登錄的暱稱：").strip()
    if not name:
        print("⚠️ 暱稱不能是空白。"); return
    code = hashlib.sha256(f"{_COURSE_NAMESPACE}|{_LEVEL}|{_norm_name(name)}".encode("utf-8")).hexdigest()[:6].upper()
    print("=" * 46)
    print(f"🎉 恭喜 {name}！{_LEVEL} 通關！")
    print(f"🔑 通關密語：{_PREFIX}-{_LEVEL}-{code}")
    print("👉 回到入口網頁，把密語貼到這一關的「輸入通關密語」欄位。")
    print("=" * 46)

try:
    import numpy as _np_, pandas as _pd_
    _np_.random.seed(42)
except Exception:
    pass
print(f"🧰 魔法工具箱已準備好！本關有 {len(_TASKS)} 個任務：{'、'.join(_TASKS)}")
print("   做完每個任務後，執行它下方的「檢查」格；全部通過後執行最下方的「通關密語」。")

# ---------------- 各任務的檢查規則 ----------------
def _check_9_1(run):
    out, ns = run()
    if tuple(抓變數(ns, "形狀")) != (2000, 6): return (False, "Z 應該是 2000 × 6。")
    if not 約等於(抓變數(ns, "第一欄平均"), 0, 1e-6): return (False, "標準化後平均應該是 0。")
    return (約等於(抓變數(ns, "第一欄標準差"), 1, 1e-3), "標準化後標準差應該是 1（Z[:, 0].std()）。")
任務定義("9-1", _check_9_1, 提示="Z[:, 0] 是第一欄。")

def _check_9_2(run):
    out, ns = run()
    e = 抓變數(ns, "解釋比例")
    if len(e) != 2 or not 約等於(e[0], 0.23713, 0.005): return (False, "解釋比例 = pca.explained_variance_ratio_（n_components=2）。")
    if not 約等於(抓變數(ns, "合計解釋比例"), 0.41206, 0.005): return (False, "合計 = 解釋比例.sum()。")
    return (str(抓變數(ns, "PC1主角")) in ("來店次數", "距上次來店天數"), "PC1 主角是活躍度相關的欄位。")
任務定義("9-2", _check_9_2, 提示="解釋比例.sum()。")

def _check_9_3(run):
    out, ns = run()
    if not run.figs: return (False, "沒有畫出圖。")
    f = run.figs[0]
    return ("PCA" in f["title"].upper() and f["n_collections"] >= 1 and f["n_patches"] >= 6, "標題含 PCA、有散佈點、六支箭頭。")
任務定義("9-3", _check_9_3, 提示="plt.title('PCA 雙標圖')。")

def _check_9_4(run):
    out, ns = run()
    s = list(抓變數(ns, "慣性們"))
    if len(s) != 8: return (False, "K 從 1 到 8，共 8 個慣性（range(1, 9)）。")
    if not 約等於(s[0], 12000.0, 5): return (False, "K=1 的慣性應該等於標準化資料的總平方和。")
    if any(s[i] < s[i + 1] for i in range(7)): return (False, "慣性應該隨 K 變大而變小。")
    return (約等於(抓變數(ns, "K2下降幅度"), 0.1438, 0.02), "K2下降幅度 = (慣性們[0] - 慣性們[1]) / 慣性們[0]。")
任務定義("9-4", _check_9_4, 提示="range(1, 9)。")

def _check_9_5(run):
    out, ns = run()
    n = 抓變數(ns, "各群人數")
    if len(n) != 4 or int(n.sum()) != 2000: return (False, "應該有 4 群、合計 2000 人。")
    avg = 抓變數(ns, "各群平均")
    ok, msg = 資料框像(avg, 列=4, 含欄位=數值欄)
    if not ok: return (False, msg)
    if int(抓變數(ns, "沉睡群")) != int(avg["距上次來店天數"].idxmax()): return (False, "沉睡群 = 距上次來店天數 平均最大的群。")
    if int(抓變數(ns, "常客群")) != int(avg["來店次數"].idxmax()): return (False, "常客群 = 來店次數 平均最大的群。")
    d = 抓變數(ns, "命名", dict)
    return (len(d) == 4 and set(map(int, d.keys())) == {0, 1, 2, 3} and all(isinstance(v, str) and v and "?" not in v for v in d.values()), "命名 要有 0–3 四個群、每群一個名字。")
任務定義("9-5", _check_9_5, 提示="常客群 = int(各群平均['來店次數'].idxmax())；其他兩群自己看表取名。")

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
members = pd.read_csv("https://raw.githubusercontent.com/johnnychao/stats-quest-2026/v1.2.1/data/coffee_members.csv")
數值欄 = ["年齡", "加入月數", "來店次數", "平均消費", "距上次來店天數", "住家距離"]
members[數值欄].describe().round(1)

## 🧭 9-1　沒有標準答案的學習
前面每一關都有 Y（回購、營收、心臟病）可以對答案。**非監督式學習**沒有 Y：只有 2,000 位會員的六個數字，老闆問「我的客人可以分成哪幾種？」——沒有標準答案，只能找**結構**。
兩把工具：
- **PCA（主成分分析）**：六個欄位太多看不清，找「最能看出形狀的角度」把它壓成 2 軸——像用手電筒把立體物投影成影子。
- **K-means（分群）**：把相近的人分到同一群——先隨機分房間，再反覆換房，直到大家都待在最像自己的那間。

⚠️ 兩者都靠「距離」，所以**一定先標準化**，否則平均消費（幾百）會壓過來店次數（幾次）。

In [ ]:
#@title 🈶 中文字型設定（畫圖前先執行；Colab 初次約 20～40 秒）
import glob, shutil, subprocess, sys, matplotlib
from matplotlib import font_manager

_font_globs = [
    '/usr/share/fonts/opentype/noto/NotoSansCJK*.ttc',
    '/usr/share/fonts/opentype/noto/NotoSansCJK*.otf',
    'C:/Windows/Fonts/msjh*.ttc',
]
if sys.platform.startswith('linux') and shutil.which('apt-get'):
    if not any(glob.glob(pattern) for pattern in _font_globs[:2]):
        try:
            subprocess.run(
                ['apt-get', '-qq', 'install', '-y', 'fonts-noto-cjk'],
                check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
            )
        except (FileNotFoundError, subprocess.CalledProcessError) as error:
            raise RuntimeError('無法自動安裝中文字型；請確認網路後重新執行本格。') from error
for pattern in _font_globs:
    for path in glob.glob(pattern):
        try:
            font_manager.fontManager.addfont(path)
        except (OSError, RuntimeError):
            pass

_available_fonts = {font.name for font in font_manager.fontManager.ttflist}
_preferred_fonts = [
    'Noto Sans TC', 'Noto Sans CJK TC',
    'Microsoft JhengHei', 'Microsoft JhengHei UI', 'PingFang TC',
    'Noto Sans CJK JP', 'Arial Unicode MS',
]
_chinese_font = next((name for name in _preferred_fonts if name in _available_fonts), None)
if _chinese_font is None:
    raise RuntimeError('找不到可顯示繁體中文的字型；請安裝 Noto Sans CJK 後重新執行本格。')
matplotlib.rcParams['font.family'] = 'sans-serif'
matplotlib.rcParams['font.sans-serif'] = [_chinese_font, 'DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False
print(f"✅ 中文字型設定完成：{_chinese_font}")

In [ ]:
Z = StandardScaler().fit_transform(members[數值欄])
pca = PCA(n_components=2).fit(Z)
print("兩個主成分各解釋的變異比例：", pca.explained_variance_ratio_.round(3), "合計", pca.explained_variance_ratio_.sum().round(3))
負荷 = pd.DataFrame(pca.components_.T, index=數值欄, columns=["PC1", "PC2"]).round(2)
print(負荷)                        # 每個原始欄位對兩個軸的「貢獻」（正負代表方向）

## 9-2　雙標圖：一張圖看兩件事
把 2,000 位會員投影到 PC1、PC2 平面（點），再把六個欄位的方向畫成箭頭：箭頭指向哪裡，那一區的會員那個數字就大。
PC1 的兩端通常是「常來 vs 很久沒來」——**活躍度**；讀圖的方法就是看箭頭。

In [ ]:
分數 = pca.transform(Z)
plt.figure(figsize=(7, 6))
plt.scatter(分數[:, 0], 分數[:, 1], s=6, alpha=.3)
for i, col in enumerate(數值欄):
    plt.arrow(0, 0, pca.components_[0, i] * 3, pca.components_[1, i] * 3, color="red", head_width=.08)
    plt.text(pca.components_[0, i] * 3.3, pca.components_[1, i] * 3.3, col, color="red")
plt.xlabel("PC1"); plt.ylabel("PC2"); plt.title("PCA 雙標圖：會員分布與六個欄位的方向"); plt.show()

## 9-3　K-means 與肘部法：分幾群？
K-means 要先說分幾群（K）。**慣性（inertia）**是每個人到自己那群中心的距離平方和，K 越大一定越小——找「下降開始變慢」的那個轉折（肘部）。沒有標準答案，通常配合「老闆用得動幾群」來決定；這一關用 K = 4。

In [ ]:
慣性們 = [KMeans(n_clusters=k, n_init=10, random_state=42).fit(Z).inertia_ for k in range(1, 9)]
plt.plot(range(1, 9), 慣性們, marker="o"); plt.xlabel("K"); plt.ylabel("慣性"); plt.title("肘部法"); plt.show()
km = KMeans(n_clusters=4, n_init=10, random_state=42).fit(Z)
members["群"] = km.labels_
print(members["群"].value_counts().sort_index())
print(members.groupby("群")[數值欄].mean().round(1))       # 幫每一群取名字，就看這張表

### 🎯 任務 9-1　標準化

把六個數值欄標準化成 `Z`（`StandardScaler().fit_transform`），確認 `Z` 的形狀 `形狀`，以及 `第一欄平均`（應接近 0）與 `第一欄標準差`（應接近 1）。

In [ ]:
# 🎯 任務 9-1　標準化（請保留這一行）
Z = StandardScaler().fit_transform(members[數值欄])
形狀 = Z.shape
第一欄平均 = Z[:, 0].mean()
第一欄標準差 = ???
print(形狀, round(第一欄平均, 6), round(第一欄標準差, 6))

In [ ]:
檢查("9-1")   # ◀ 執行這一格，看看任務 9-1 有沒有過關

### 🎯 任務 9-2　PCA

對 `Z` 做 `PCA(n_components=2)`，存成 `pca`；取出 `解釋比例`（兩個數的陣列）、`合計解釋比例`，並把對 PC1 貢獻絕對值最大的欄位名存成 `PC1主角`。

In [ ]:
# 🎯 任務 9-2　PCA（請保留這一行）
pca = PCA(n_components=2).fit(Z)
解釋比例 = pca.explained_variance_ratio_
合計解釋比例 = ???
PC1主角 = 數值欄[int(np.argmax(np.abs(pca.components_[0])))]
print(解釋比例.round(3), round(合計解釋比例, 3), PC1主角)

In [ ]:
檢查("9-2")   # ◀ 執行這一格，看看任務 9-2 有沒有過關

### 🎯 任務 9-3　雙標圖

畫出雙標圖：`pca.transform(Z)` 的散佈點，加上六個欄位的箭頭與文字，標題要包含「PCA」。

In [ ]:
# 🎯 任務 9-3　雙標圖（請保留這一行）
分數 = pca.transform(Z)
plt.figure(figsize=(7, 6))
plt.scatter(分數[:, 0], 分數[:, 1], s=6, alpha=.3)
for i, col in enumerate(數值欄):
    plt.arrow(0, 0, pca.components_[0, i] * 3, pca.components_[1, i] * 3, color="red", head_width=.08)
    plt.text(pca.components_[0, i] * 3.3, pca.components_[1, i] * 3.3, col, color="red")
plt.xlabel("PC1"); plt.ylabel("PC2"); plt.title(???); plt.show()

In [ ]:
檢查("9-3")   # ◀ 執行這一格，看看任務 9-3 有沒有過關

### 🎯 任務 9-4　肘部法

對 K = 1～8 各跑一次 `KMeans(n_clusters=k, n_init=10, random_state=42)`，把慣性存成 `慣性們`（8 個），並算 `K2下降幅度`（K=1 到 K=2 慣性減少的比例）。

In [ ]:
# 🎯 任務 9-4　肘部法（請保留這一行）
慣性們 = [KMeans(n_clusters=k, n_init=10, random_state=42).fit(Z).inertia_ for k in range(1, ???)]
K2下降幅度 = (慣性們[0] - 慣性們[1]) / 慣性們[0]
plt.plot(range(1, 9), 慣性們, marker="o"); plt.title("肘部法"); plt.xlabel("K"); plt.show()
print(np.round(慣性們, 0), round(K2下降幅度, 3))

In [ ]:
檢查("9-4")   # ◀ 執行這一格，看看任務 9-4 有沒有過關

### 🎯 任務 9-5　分四群並取名

用 `KMeans(n_clusters=4, n_init=10, random_state=42)` 分群，把標籤存進 `members['群']`；算出 `各群人數`（value_counts）與 `各群平均`（groupby 群 的六欄平均）；找出**最久沒來**的群 `沉睡群`（距上次來店天數平均最大）與**最常來**的群 `常客群`；最後給四群取名字，存成字典 `命名`（key 為群編號 0–3、value 為名字）。

In [ ]:
# 🎯 任務 9-5　分四群並取名（請保留這一行）
km = KMeans(n_clusters=4, n_init=10, random_state=42).fit(Z)
members["群"] = km.labels_
各群人數 = members["群"].value_counts().sort_index()
各群平均 = members.groupby("群")[數值欄].mean().round(1)
沉睡群 = int(各群平均["距上次來店天數"].idxmax())
常客群 = ???
命名 = {沉睡群: "沉睡會員", 常客群: "常客", ???: "???", ???: "???"}
print(各群人數); print(各群平均); print(命名)

In [ ]:
檢查("9-5")   # ◀ 執行這一格，看看任務 9-5 有沒有過關

## 🌟 進階挑戰（不計分）
1. 把「群」當成新特徵加進 Day 3 的回購模型（one-hot），AUC 有沒有提高？（最終 Boss 的進階挑戰會用到）
2. 用 K = 3 與 K = 5 再分一次，哪一種老闆比較用得動？

---
## 🔑 通關密語
　你已經能在沒有標準答案的資料裡找出結構，並用人話幫每一群取名字。
全部任務都 ✅ 之後，執行下面這一格，會得到你專屬的通關密語（和暱稱綁定，每個人不一樣）。

In [ ]:
通關密語()

---
### 🧭 接下來
**下一關：👑 FINAL 最終 Boss：勇者咖啡 2.0 兩份預測報告** → [在 Colab 開啟](https://colab.research.google.com/github/johnnychao/stats-quest-2026/blob/v1.2.1/notebooks/FINAL_boss_coffee2.ipynb)

回到入口網頁：https://johnnychao.github.io/stats-quest-2026/